# Phase 2 — Roma's Track (C1, C2, C3) — GPU Runtime

Runs your Cross-Lingual Fallback, Sufficiency Labeling, and Query Refinement code.

**Before running:** Runtime → Change runtime type → Hardware accelerator → **GPU (T4)** → Save.

C2 benefits most from GPU (it can use Phase 1's real embedding model to score sufficiency far more precisely than the lexical fallback). C1 and C3 don't need GPU themselves but run fine alongside it.


## 1. Confirm GPU is active

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only - go back and enable GPU in Runtime settings")


CUDA available: True
Device: Tesla T4


## 2. Upload the code

In [ ]:
from google.colab import files

print("Select Roma_Phase2_C1_C2_C3_Code.zip")
uploaded = files.upload()


Select Roma_Phase2_C1_C2_C3_Code.zip


Saving Roma_Phase2_C1_C2_C3_Code.zip to Roma_Phase2_C1_C2_C3_Code.zip


In [ ]:
!unzip -o -q Roma_Phase2_C1_C2_C3_Code.zip -d Phase2_Roma
%cd Phase2_Roma
!find . -type f


/content/Phase2_Roma
./requirements.txt
./README.md
./src/c2_sufficiency_labels.py
./src/c1_cross_lingual_fallback.py
./src/c3_query_refinement.py


## 3. Install dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q sentence-transformers hnswlib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 50.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 4. C1 — Cross-lingual fallback resources
Downloads AyaTEC (real Arabic Quranic QA) and a SQuAD v2 sample (English QA).

In [ ]:
!python src/c1_cross_lingual_fallback.py


[STEP] Downloading AyaTEC from https://sites.google.com/view/bigir/ayatec_v1.2.zip ...
[WARN] Automatic AyaTEC download failed (404 Client Error: Not Found for url: https://sites.google.com/view/bigir/ayatec_v1.2.zip). Download it manually from https://sites.google.com/view/bigir/datasets and place it at ./ayatec_v1.2.zip, then re-run this script.
[STEP] Downloading SQuAD v2 sample...
README.md: 100% 8.92k/8.92k [00:00<00:00, 21.1MB/s]

squad_v2/train-00000-of-00001.parquet: downloading bytes:  29% 4.74M/16.4M [00:01<00:01, 6.64MB/s, 30.8kB/s  ]
squad_v2/train-00000-of-00001.parquet: downloading bytes: 100% 15.9M/15.9M [00:01<00:00, 11.7MB/s, 1.55MB/s  ]
squad_v2/train-00000-of-00001.parquet: reconstructing file: 100% 16.4M/16.4M [00:01<00:00, 12.0MB/s, 1.59MB/s  ]

squad_v2/validation-00000-of-00001.parqu(…): downloading bytes:  80% 1.07M/1.35M [00:00<00:00, 1.93MB/s, 5.02kB/s  ]
squad_v2/validation-00000-of-00001.parqu(…): downloading bytes: 100% 1.30M/1.30M [00:00<00:00, 1.73MB/s,  

If AyaTEC's automatic download fails (the host sometimes serves an interstitial page instead of the raw zip), download it manually and upload it here:

In [ ]:
import os
if not os.path.exists("ayatec_v1.2.zip"):
    print("AyaTEC auto-download may have failed. If so, get it from:")
    print("https://sites.google.com/view/bigir/datasets")
    print("then run this cell to upload it manually:")
    from google.colab import files
    uploaded_ayatec = files.upload()  # select ayatec_v1.2.zip if you downloaded it manually
    if uploaded_ayatec:
        !python src/c1_cross_lingual_fallback.py  # re-run now that the zip is present


## 5. Get Member A's real QRCD data (needed for C2)

C2 needs `data/qrcd_flat.json`. Upload it if you have it, or regenerate it directly.

In [ ]:
import os
os.makedirs("data", exist_ok=True)

if not os.path.exists("data/qrcd_flat.json"):
    print("Regenerating QRCD data directly...")
    import json, requests

    records = []
    urls = {
        "train": "https://raw.githubusercontent.com/RanaMalhas/QRCD/main/dataset/qrcd_v1.1_train.json",
        "test": "https://raw.githubusercontent.com/RanaMalhas/QRCD/main/dataset/qrcd_v1.1_test.json",
    }
    for split_name, url in urls.items():
        raw = requests.get(url, timeout=60).json()
        for para in raw["data"]:
            passage = para["paragraphs"][0]["context"]
            for qa in para["paragraphs"][0]["qas"]:
                records.append({
                    "split": split_name, "pq_id": qa["id"], "passage": passage,
                    "question": qa["question"], "answers": [a["text"] for a in qa.get("answers", [])],
                })

    with open("data/qrcd_flat.json", "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    print(f"Saved {len(records)} QRCD records to data/qrcd_flat.json")
else:
    print("data/qrcd_flat.json already present.")


## 6. C2 — Sufficiency-labeled dataset (fast version, no GPU needed)

Quick run using the lexical-overlap proxy — works immediately, weaker signal.

In [ ]:
!python src/c2_sufficiency_labels.py --sample-size 100


## 7. C2 (better) — Real GPU-accelerated scoring using Phase 1's model

This uses your actual fine-tuned embedding model for far more accurate sufficiency scores. Needs Phase 1's `b5_real_finetuned` model and `index/` folder from Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Adjust this path to wherever you saved Phase 1's outputs
PHASE1_DIR = "/content/drive/MyDrive/Phase1_Project/MemberB_B4_B6_output"
!ls "{PHASE1_DIR}"


In [ ]:
import shutil, os

shutil.copytree(f"{PHASE1_DIR}/b5_real_finetuned", "b5_real_finetuned", dirs_exist_ok=True)
shutil.copytree(f"{PHASE1_DIR}/index", "index", dirs_exist_ok=True)

if not os.path.exists("src/b6_build_index_and_retrieval_api.py"):
    from google.colab import files
    print("Select b6_build_index_and_retrieval_api.py")
    uploaded_b6 = files.upload()
    shutil.move("b6_build_index_and_retrieval_api.py", "src/b6_build_index_and_retrieval_api.py")

print("Ready.")


In [ ]:
import sys
sys.path.insert(0, "src")

from sentence_transformers import SentenceTransformer
from b6_build_index_and_retrieval_api import load_index, RetrievalAPI
import json

model = SentenceTransformer("./b5_real_finetuned")  # runs on GPU automatically if available
index, entries = load_index(dim=model.get_sentence_embedding_dimension(), out_dir="index")
retrieval_api = RetrievalAPI(model, index, entries)
print(f"Loaded real retrieval API with {len(entries)} indexed entries (GPU: {torch.cuda.is_available()}).")

from c2_sufficiency_labels import load_qrcd, build_labels_with_real_retrieval, verify_labels, save_labels

qrcd_records = load_qrcd()[:100]
labeled_real = build_labels_with_real_retrieval(qrcd_records, retrieval_api)
verify_labels(labeled_real)
save_labels(labeled_real)


## 8. C3 — Query refinement logic (rule-based, no GPU needed)

In [ ]:
!python src/c3_query_refinement.py


## 9. Save everything to Google Drive

In [ ]:
DEST = "/content/drive/MyDrive/Phase2_Project/Roma_output"
!mkdir -p "{DEST}"
!cp -r data "{DEST}/" 2>/dev/null
!cp -r quranNLP "{DEST}/" 2>/dev/null
!cp -r src "{DEST}/"
print(f"Saved to: {DEST}")
!find "{DEST}" -maxdepth 2


## 10. Sync Point — hand off to Laiba

The three files Laiba needs:
- `quranNLP/shared/data/ayatec_records.json` + `squad_v2_sample.json` (C1)
- `data/sufficiency_labels.json` (C2 — prefer the GPU-scored version from step 7 if you ran it)
- `src/c3_query_refinement.py` (C3 — she imports `refine_query` directly)

All saved to Drive at the path above.